# SOX32 enhancer optimization

Can the experimentally active zebrafish SOX32 enhancer be made more DE-specific according to the 22-output multitask accessibility model? This notebook interprets cached evolution results; GPU computation remains in the numbered scripts.

The design follows the [CREsted sequence-evolution tutorial](https://crested.readthedocs.io/en/latest/tutorials/model_training_and_eval.html#sequence-evolution), but starts from the tested 2,114-bp construct and permits substitutions only in the 1,100-bp enhancer.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'tables').exists():
    raise FileNotFoundError('Run this notebook from the experiment root after scripts/02_analyze.py')
TABLES = ROOT / 'tables'
FIGURES = ROOT / 'figures'
RESULTS = ROOT / 'results'

trajectory = pd.read_csv(TABLES / 'trajectory_metrics.tsv', sep='\t')
candidates = pd.read_csv(TABLES / 'candidate_summary.tsv', sep='\t')
mutations = pd.read_csv(TABLES / 'candidate_mutations.tsv', sep='\t')
pareto = pd.read_csv(TABLES / 'pareto_steps.tsv', sep='\t')

## 1. Preconditions and provenance

The preflight must reproduce the June WT prediction, confirm the 2,114-bp input and 22-output model shapes, and verify that only positions 507–1606 are mutable.

In [ ]:
with open(RESULTS / 'preflight.json') as handle:
    preflight = json.load(handle)
with open(RESULTS / 'run_metadata.json') as handle:
    run_metadata = json.load(handle)
display(pd.Series({
    'preflight': preflight['status'],
    'CREsted': run_metadata['crested_version'],
    'model SHA256': run_metadata['model_sha256'],
    'construct SHA256': run_metadata['sequence_sha256'],
    'mutable interval': run_metadata['enhancer_interval_0_based_half_open'],
    'steps per objective': run_metadata['n_mutations'],
}))

## 2. Wild-type baseline

The key baseline is not merely whether DE is the largest output, but its margin over the strongest non-DE output.

In [ ]:
wt = candidates.query("candidate_id == 'SOX32_WT'")
display(wt.T)

## 3. Evolution trajectories

DE activity is a positive control for raw output. CREsted default subtracts the summed off-target change scaled by the total number of model outputs. The custom DE-specificity trajectory additionally targets the strongest off-target and discourages DE from falling below WT.

In [ ]:
display(Image(filename=str(FIGURES / 'fig1_trajectories.png')))
display(trajectory.groupby('objective').tail(1)[[
    'objective', 'step', 'net_substitutions', 'target_prediction',
    'max_other_prediction', 'max_other_celltype', 'target_vs_max_log2', 'target_rank'
]])

## 4. Activity–specificity tradeoff

Upper-left movement is desirable: more DE accessibility with less accessibility in the strongest off-target. The Pareto table retains steps for which no other tested step has both higher DE and lower maximum off-target.

In [ ]:
display(Image(filename=str(FIGURES / 'fig2_activity_specificity.png')))
display(pareto[[
    'objective', 'step', 'net_substitutions', 'target_prediction',
    'max_other_prediction', 'max_other_celltype', 'target_vs_max_log2'
]])

## 5. Candidate snapshots across all 22 outputs

The heatmap is relative to WT. A good candidate should show a DE-selective increase or an off-target-selective decrease rather than a uniform shift across the trajectory.

In [ ]:
display(Image(filename=str(FIGURES / 'fig3_candidate_heatmap.png')))
display(candidates.sort_values(['objective', 'budget']))

## 6. Before and after across cell types

WT predictions are shown as the cell-type-colored bars and each optimized specificity candidate as the black point-and-line overlay. Separate full-size panels are saved for steps 1, 5, 10, and 20.

In [ ]:
display(Image(filename=str(FIGURES / 'fig5_before_after_specificity_grid.png')))

## 7. Mutation burden and positions

CREsted can revisit a position, so net substitutions can be lower than the step number. Candidate ordering should use the net mutation table and full FASTA rather than assuming one retained substitution per step.

In [ ]:
display(Image(filename=str(FIGURES / 'fig4_mutation_map.png')))
display(mutations)

## 8. Result and selection guardrails

The 20-mutation specificity design raises predicted DE from 169.1 to 911.2 and the DE-to-strongest-off-target ratio from 1.52 to 3.43. The 10-mutation design is a useful compromise (DE 748.6; ratio 3.24). The first specificity mutation alone, construct T1322A (enhancer base 815), leaves DE nearly unchanged while lowering the strongest off-target from 111.4 to 65.9.

These gains are relative: the step-5/10/20 designs increase DE more strongly than they increase off-target accessibility, but do not lower the absolute maximum off-target below WT. Treat them as model-nominated constructs, not evidence of reporter activity. Prioritize a small Pareto-spanning set, retain WT and a DE-activity control, and review whether substitutions create or destroy plausible EOMES/T, MIXL1, SOX17, FOXH1, or late endocrine motifs before synthesis. Predictions should also be checked under reverse complement and, where practical, with an independently trained model.